# Fundamentals of Social Data Science
# Week 5 Day 2. LLM annotation / LLM-as-judge 

> SPOILER ALERT! We will be classifying spoilers today. 
>
> This is using the Movie stack from 2024 that's on the server so it won't be especially up to date, but if you haven't yet learned that Darth Vader is actually Luke's father, I have some bad news for you.

In this worksheet we will be using an API to send queries to a server and receive responses back. This is a common pattern in the usage of LLMs. In the case, you will be sending queries to a local model that is hosted on brains.oii.ox.ac.uk. In order for this code to work, you should be able to ensure that you have port forwarding on since the server is behind a firewall. 

Once you log on to the VPN, then you should 
1. Open a remote session to brains.oii.ox.ac.uk. 
2. Then go to the terminal and navigate to `/data/$user/` (i.e. `cd /data/$user/`). 
3. From there you can clone the repo on Github: `git clone https://github.com/berniehogan/fsds26-w5d1`. This should bring down the repo for you.
4. Navigate into that folder (`cd fsds26-wk5d1`)
5. Then reopen VS Code as a project in that folder (`code -r .`)
6. Now in the terminal type `uv sync`  and this should sort out your project.

## Part 1. Road testing the server

We will be doing this in a few steps. The first step will be to confirm that you can send a query to the server. The code below uses the OpenAI API (not sending directly to OpenAI, just using their library, which is common). You should be able to run this immediately. 

Some things to point out in the code: 
- We create a client using `OpenAI(base_url=...)`. I have not set an API key but generally if sending to a remote model you would be expected to set a key here to authenticate and monitor token usage. 
- We select our model (which happens to be the only model running on vLLM). 
- The response in this case reports back the top 5 options. Given that we asked "Answer yes or no" then "yes" or "Yes" should be the first answer. Notice that it also sends down "**" as one of the top 5 tokens. Why might that be? 

In [ ]:
import json, math
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")
MODEL = "gemma-4-31b"
ask = [{"role": "user", "content":
        'Does this sentence reveal a plot twist? Answer yes or no.\n\n"Bruce Willis was dead the whole time."'}]

# 1. Probabilities for the first token of the answer
r = client.chat.completions.create(model=MODEL, messages=ask, max_tokens=1, temperature=0,
                                   logprobs=True, top_logprobs=5)

for t in r.choices[0].logprobs.content[0].top_logprobs:
    print(f"{t.token!r:10} {math.exp(t.logprob):.3f}")


Now below we will see how we can have the output constrained to a specific format, specifically JSON, which is common and will be highly useful for many structured tasks. See how we structure "response_format" below. 

In [ ]:
# 2. Output forced to match a JSON schema
schema = {"type": "object",
          "properties": {"reason": {"type": "string"}, "spoiler": {"type": "boolean"}},
          "required": ["reason", "spoiler"]}
r = client.chat.completions.create(model=MODEL, messages=ask, temperature=0, max_tokens=200,
    response_format={"type": "json_schema", "json_schema": {"name": "judgement", "schema": schema}})

print(json.loads(r.choices[0].message.content))

Some models are 'thinking' models which mean they use 'chain of thought' in order to structure their thoughts prior to sending the official response. However, CoT is still itself a response. This we can also review the thinking trace. Notice how I also increased max_tokens. This is because thinking mode tends to eat more tokens. Some models cannot disable the thinking trace. If you run Qwen, for example, you might get a thinking trace that outruns a small number of tokens even if the desired response is merely "yes" or "no". 

In [ ]:
# 3. Thinking switched on for this one request
r = client.chat.completions.create(model=MODEL, messages=ask, max_tokens=1500,
    extra_body={"chat_template_kwargs": {"enable_thinking": True}})
msg = r.choices[0].message
print("THINKING:", (getattr(msg, "reasoning", None) or getattr(msg, "reasoning_content", None) or "")[:500])
print("ANSWER:", msg.content)
print("tokens used:", r.usage.completion_tokens)

## Part 2. Preparing our data

Normally we might want to annotate a small amount of data and cross check that with our model. However, in this case, we will use a natural labelling. The Movies stack exchange makes use of a 'spoiler' tag. Thus, we can detect which posts (questions or answers) contain a spoiler. There is one logical caution here. In statistics we have the notion of type 1 and type 2 errors. Type 1 errors are false positives (FP) and Type 2 errors are false negatives (FN). In our case, it is likely that a post with a spoiler tag is a positive. But an unlabelled post might also contain a spoiler. In a real circumstance we would want to label a sample of data, which would include both positive and negative instances. 

One issue with this labelling is "class imbalance". As we will see below, there are many, many more posts without a spoiler tag than with a spoiler tag. This means that any classification might be very much distorted by one of the classes. 

The first part of this exercise is boilerplate that you will be able to recognise since we are just loading the data and parsing it. It might take a couple seconds to load.

In [ ]:
from pathlib import Path
import py7zr
import pandas as pd

ARCHIVE_DIR = Path("/data/resource/stack_exchange_archives")
SITE = "movies"          # any <site>.stackexchange.com.7z in ARCHIVE_DIR

def find_project_root(start=None):
    """Walk up from the current folder to the one holding pyproject.toml."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError(f"No pyproject.toml found in {start} or any folder above it")

ARCHIVE = ARCHIVE_DIR / f"{SITE}.stackexchange.com.7z"
FOLDER = find_project_root() / "data" / SITE      # where the unzipped files go
FOLDER.mkdir(parents=True, exist_ok=True)

# Unzip only Posts.xml, and only if it isn't already there.
if not (FOLDER / "Posts.xml").exists():
    with py7zr.SevenZipFile(ARCHIVE, mode="r") as archive:
        archive.extract(path=FOLDER, targets=["Posts.xml"])

# Each file has one <row .../> per record, with the fields as attributes.
stack_df = pd.read_xml(FOLDER / "Posts.xml", xpath="//row")

print(stack_df.shape)
print(stack_df.columns)

Now with the data loaded, let's create a second column for BodyText as before: 

In [ ]:
import re, warnings
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

URL_RE = re.compile(r"(https?://|www\.)\S+")

def html_to_text(html):
    """HTML post body -> plain text with no links, code or images."""
    if not isinstance(html, str) or not html.strip():
        return ""
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["pre", "code", "img"]):
        tag.decompose()                       # drop these entirely
    for a in soup.find_all("a"):
        if URL_RE.match(a.get_text(strip=True)):
            a.decompose()                     # link text is just a URL: drop it
        else:
            a.unwrap()                        # keep the words, lose the link
    text = soup.get_text(" ", strip=True)
    text = URL_RE.sub("", text)               # bare URLs typed into the text
    text = re.sub(r"\s+([.,;:!?)])", r"\1", text)
    return re.sub(r"\s+", " ", text).strip()

stack_df["BodyText"] = stack_df["Body"].map(html_to_text)

Finally, for our pre-processing, we will create a 'spoiler' category based on the text. Note that this can in theory happen before `BodyText` since we extract it from `Body`. Why is that? Because it's likely to be stripped out from `BodyText`. Below the processing, notice that we do a little check on our data. Such sanity checks are extremely important after any such processing. 

You should be able to see that we do have a considerable number of spoilers but that they are still a mere fraction of the total number of posts, so if we classify these using all the data we might have a 'needle in a haystack' problem. So to that end, we will instead find a reasonable way to create a sample set for comparison.

In [ ]:
stack_df["has_spoiler"] = stack_df["Body"].str.contains('class="spoiler"', na=False)

kind = stack_df["PostTypeId"].map({1: "question", 2: "answer"})
print(stack_df.groupby(kind)["has_spoiler"].agg(posts="size", marked="sum", share="mean"), "\n")

year = pd.to_datetime(stack_df["CreationDate"]).dt.year
print(stack_df.groupby(year)["has_spoiler"].mean().round(3).to_string(), "\n")

ex = stack_df[stack_df["has_spoiler"]].sample(2, random_state=0)
for _, row in ex.iterrows():
    print(row["Body"][:400], "\n\n", row["BodyText"][:300], "\n")

### Part 2.1. Creating our annotation data

We will want to split "ground truth" data into training and test data. The goal here is to check whether there are detectable differences between the 'spoiler' data versus 'non-spoiler' data that a model might pick up on having nothing to do with the semantics of the post. Below we will first simply sample from the non-spoiler data. But notice that when we do that we will have some really unbalanced data. We can use non-parametric t-tests (Mann Whitney U) to compare differences between the two distributions. 

The code is organised in a few blocks that you can check one by one. 

Block 1. 
- Create the classification using a sample. We will treat the 'spoiler' as the ground truth / gold standard.

Block 2. 
- We simply check that we have good data integrity with equal spoiler/non-spoiler rows. 

Block 3. 
- In this block we have the code for running a small model ad hoc to measure perplexity. I've disabled this by default since we are all doing this in parallel. However, on your own time you might want to set `perplexity_run = True` and add that to the DataFrame. 

Block 4. 
- Here we calculate differences between the two samples. If the p-val is low that means that we reject the null for differences between the sample (which is not good). We want the classification to not be about simple word differences. Since the p-value will almost certainly be significant, we can see below how I created a matched sample. 

Block 5. 
- Here we look at the mean centroid of the embedding distance between the two samples. It is not guaranteed that these would be the same since they could have different content, but we might want to treat this as advisory that even if we fix the number of words / characters then we still might have some semantic differences on account of the nature of the content. This one uses SBERT but it should not be heavy on the server if it is done in quasi-parallel. 
- The code also checks the AUC between the two groups using a logistic regression. We will just observe this here as a sort of black box, but in ML this sort of approach will be unpacked in greater depth. The relevant thing is that AUC should be close to .5 (meaning 50/50 in our case) but any number below 0.65 should be tolerable for a balanced sample. 

Block 6. 
- In this block we look at the different distributions of top tags between spoilers and non-spoilers. As we can see there are many more spoilers concentrated in a few tags. For this reason below I will then consider matching by tags as well as word counts. 

In [ ]:
# Block 1. 
from sklearn.model_selection import train_test_split

q = stack_df[(stack_df["PostTypeId"] == 1) & stack_df["BodyText"].str.len().between(150, 1500)]
gold = pd.concat([q[q["has_spoiler"]].sample(100, random_state=1),
                  q[~q["has_spoiler"]].sample(100, random_state=1)])
gold = gold[["Id", "Title", "BodyText", "has_spoiler"]].reset_index(drop=True)

dev, test = train_test_split(gold, test_size=0.5, stratify=gold["has_spoiler"], random_state=0)
dev, test = dev.assign(split="dev"), test.assign(split="test")

# # A blind sheet for hand annotation: shuffled, with no label column
# (dev.sample(frac=1, random_state=3)[["Id", "Title", "BodyText"]]
#     .assign(my_label="")
#     .to_csv(FOLDER / "annotate_me.csv", index=False))



In [ ]:
# Block 2. 
q = stack_df[(stack_df["PostTypeId"] == 1) & stack_df["BodyText"].str.len().between(150, 1500)]
pos = q[q["has_spoiler"]]
check = pd.concat([pos, q[~q["has_spoiler"]].sample(len(pos), random_state=1)]).reset_index(drop=True)
y = check["has_spoiler"].to_numpy()
text = check["BodyText"]
print(check["has_spoiler"].value_counts())

In [ ]:
# Block 3. Simple model to measure perplexity 
#          If you  

perplexity_run = False 

if perplexity_run: 
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
    os.environ["CUDA_VISIBLE_DEVICES"] = "2"         # a free L40, not the GPU running vLLM

    import torch, numpy as np
    from transformers import AutoTokenizer, AutoModelForCausalLM

    PPL_MODEL = "Qwen/Qwen3-0.6B-Base"
    tok = AutoTokenizer.from_pretrained(PPL_MODEL)
    lm = AutoModelForCausalLM.from_pretrained(PPL_MODEL, dtype=torch.bfloat16).to("cuda").eval()

    @torch.no_grad()
    def perplexity(texts, max_tokens=512):
        out = []
        for t in texts:
            ids = tok(t, return_tensors="pt", truncation=True, max_length=max_tokens).input_ids.to("cuda")
            out.append(float(torch.exp(lm(ids, labels=ids).loss)))
        return np.array(out)

In [ ]:
# Block 4. Testing for simple differences between spoiler and non-spoiler text. 

from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score
import numpy as np

q = stack_df[(stack_df["PostTypeId"] == 1) & stack_df["BodyText"].str.len().between(150, 1500)]
pos = q[q["has_spoiler"]]
check = pd.concat([pos, q[~q["has_spoiler"]].sample(len(pos), random_state=1)]).reset_index(drop=True)
y = check["has_spoiler"].to_numpy()
text = check["BodyText"]
print(check["has_spoiler"].value_counts())

if perplexity_run: 
    features = pd.DataFrame({
        "characters": text.str.len(),
        "words": text.str.split().str.len(),
        "colons per 100 words": 100 * text.str.count(":") / text.str.split().str.len(),
        "perplexity": perplexity(text.tolist()),
    })
else:
    features = pd.DataFrame({
            "characters": text.str.len(),
            "words": text.str.split().str.len(),
            "colons per 100 words": 100 * text.str.count(":") / text.str.split().str.len()
        })

def feature_report(features, y):
    rows = []
    for name, x in features.items():
        x = np.asarray(x, float)
        rows.append({"feature": name,
                     "median (spoiler)": np.median(x[y]), "median (other)": np.median(x[~y]),
                     "AUC": roc_auc_score(y, x),
                     "p (Mann-Whitney)": mannwhitneyu(x[y], x[~y]).pvalue})
    return pd.DataFrame(rows).round(3)

feature_report(features, y)

In [ ]:
# Block 5. 

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold

sbert = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
emb = sbert.encode(text.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=True)

S = emb @ emb.T
off = ~np.eye(len(y), dtype=bool)
print(f"mean cosine  spoiler-spoiler {S[np.ix_(y, y)][off[np.ix_(y, y)]].mean():.3f}"
      f"   other-other {S[np.ix_(~y, ~y)][off[np.ix_(~y, ~y)]].mean():.3f}"
      f"   spoiler-other {S[np.ix_(y, ~y)].mean():.3f}")

c_pos, c_neg = emb[y].mean(0), emb[~y].mean(0)
print(f"cosine between the two group centroids: {c_pos @ c_neg / np.linalg.norm(c_pos) / np.linalg.norm(c_neg):.3f}")

p = cross_val_predict(LogisticRegression(max_iter=2000), emb, y,
                      cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
print(f"AUC of a classifier on the embeddings: {roc_auc_score(y, p):.3f}")

In [ ]:
# Block 6. Investigating spoiler bias by tag 

tags = check["Tags"].str.strip("|<>").str.split(r"\|+|><")
share = lambda m: tags[m].explode().value_counts(normalize=True)
top = pd.DataFrame({"spoiler": share(y), "other": share(~y)}).fillna(0)
top["ratio"] = (top["spoiler"] + 0.005) / (top["other"] + 0.005)
top.sort_values("spoiler", ascending=False).head(15).round(3)

## Creating a balanced sample 

A balanced sample is not strictly necessary but it will help us consider whether the spoiler labeller is picking up on semantic differences or just difference in content. For example, if it is biased to assume that Star Wars or Game of Thrones is more 'spoiler-oriented' then this might otherwise undermine our annotation detection. 

To note, the point of this is to check that the spoiler labelling is good. If we believe it is good with this approach then we can let it run on _all_ of the text. It is not necessary for us to have our labelling balanced between positive and negative, this is for us to convince someone else that the labeller works. Then running it on all of the text is just a matter of running the prompt across all text. But to note, if the accuracy is relatively low then it might lead to either many false positive or false negatives on the data. A more complex sort of query is likely to be noisier. For example, classifying questions versus answers should be a lot more accurate. 

Note that the code below is AI-steered. It is still readable and as such I would recommend that you read it line by line to see what is happenening and what you can appreciate here. 

- Block 1. Create the matched sample, stratified by tags and within roughly the same length. 
- Block 2. Checking that there are no significant differences in words. This should be good now. A really high p-value means that there's no significant difference, which will make our assessment of the LLM annotation stronger. 

In [ ]:
# Block 1. Create stratified sample.

qs = q.copy()                      # questions in the 150-1,500 character band, as before
tag_lists = qs["Tags"].fillna("").str.strip("|<>").str.split(r"\|+|><")
freq = tag_lists.explode().value_counts()

def stratum(tags, min_count=20):
    """The most specific tag that still has at least min_count questions (usually the show or film)."""
    ok = [t for t in tags if t and freq[t] >= min_count]
    return min(ok, key=freq.get) if ok else max(tags, key=lambda t: freq.get(t, 0))

qs["stratum"] = tag_lists.map(stratum)
qs["loglen"] = np.log(qs["BodyText"].str.len())

pool = qs[~qs["has_spoiler"]].copy()
pairs = []
for i, row in qs[qs["has_spoiler"]].sample(frac=1, random_state=0).iterrows():
    cand = pool[pool["stratum"] == row["stratum"]]
    if cand.empty:
        continue
    gap = (cand["loglen"] - row["loglen"]).abs()
    j = gap.idxmin()
    if gap[j] > 0.2:                       # lengths within about 20% of each other
        continue
    pairs.append((i, j))
    pool = pool.drop(j)                    # each unmarked post is used only once

matched = pd.concat([qs.loc[[i for i, _ in pairs]].assign(pair=range(len(pairs))),
                     qs.loc[[j for _, j in pairs]].assign(pair=range(len(pairs)))]).reset_index(drop=True)
print(f"matched {len(pairs)} of {qs['has_spoiler'].sum()} marked posts")
print(matched[matched["has_spoiler"]]["stratum"].value_counts().head(10))

In [ ]:
# Block 2. Checking the sample for differences.

check = matched
y = check["has_spoiler"].to_numpy()
text = check["BodyText"]

features = pd.DataFrame({
    "characters": text.str.len(),
    "words": text.str.split().str.len(),
    "colons / 100 words": 100 * text.str.count(":") / text.str.split().str.len(),
})

if perplexity_run: 
    features["perplexity"] = perplexity(text.tolist())

display(feature_report(features, y))

emb = sbert.encode(text.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=True)
p = cross_val_predict(LogisticRegression(max_iter=2000), emb, y,
                      cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
print(f"AUC of a classifier on the embeddings: {roc_auc_score(y, p):.3f}")

In [ ]:
dev_pairs, test_pairs = train_test_split(
    matched["pair"].unique(), test_size=0.5, random_state=0
)

gold = matched[["Id", "Title", "BodyText", "has_spoiler", "pair"]].copy()
gold["split"] = np.where(gold["pair"].isin(dev_pairs), "dev", "test")
gold.to_csv(FOLDER / "spoiler_gold.csv", index=False)


## Part 3. Doing the annotations. 

Now below we are doing the actual annotation. For this, we can use the local model that we have downloaded. 

In [ ]:
import math, time
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from sklearn.metrics import confusion_matrix, roc_auc_score

client = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")
MODEL = "gemma-4-31b"

gold = pd.read_csv(FOLDER / "spoiler_gold.csv")
dev = gold[gold["split"] == "dev"].reset_index(drop=True)

PROMPT_V1 = """Does the following post from a film and TV question-and-answer site reveal a major plot point? Answer yes or no."

Title: {title}

{body}"""

def ask(row, template):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": template.format(title=row.Title, body=row.BodyText)}],
        max_tokens=1, temperature=0, logprobs=True, top_logprobs=20)
    top = r.choices[0].logprobs.content[0].top_logprobs
    p_yes = sum(math.exp(t.logprob) for t in top if t.token.strip().lower() == "yes")
    p_no = sum(math.exp(t.logprob) for t in top if t.token.strip().lower() == "no")
    return {"first_token": r.choices[0].message.content,
            "p_yes": p_yes / (p_yes + p_no) if p_yes + p_no > 0 else np.nan}

def run(df, template=PROMPT_V1, workers=32):
    t0 = time.time()
    with ThreadPoolExecutor(workers) as pool:
        out = list(pool.map(lambda row: ask(row, template), df.itertuples()))
    print(f"{len(df)} posts in {time.time() - t0:.0f} s")
    return df.assign(**pd.DataFrame(out, index=df.index))

def score(res, threshold=0.5):
    ok = res["p_yes"].notna()
    y = res.loc[ok, "has_spoiler"].astype(bool).to_numpy()
    pred = (res.loc[ok, "p_yes"] >= threshold).to_numpy()
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[False, True]).ravel()
    precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    print(f"no yes/no answer: {(~ok).sum()} of {len(res)}")
    print(f"first token not yes/no: {(~res['first_token'].str.strip().str.lower().isin(['yes', 'no'])).sum()}")
    print(f"\n                 marked   unmarked")
    print(f"model says yes   {tp:6d}   {fp:8d}")
    print(f"model says no    {fn:6d}   {tn:8d}\n")
    print(f"precision {precision:.2f}   recall {recall:.2f}   F1 {2 * precision * recall / max(precision + recall, 1e-9):.2f}"
          f"   accuracy {(tp + tn) / len(y):.2f}   AUC of P(yes) {roc_auc_score(y, res.loc[ok, 'p_yes']):.3f}")

res_v1 = run(dev)
res_v1.to_csv(FOLDER / "results_v1_zeroshot.csv", index=False)
score(res_v1)

# To run on all of the data instead: (it will be twice as slow)
# res_v2 = run(gold)
# res_v2.to_csv(FOLDER / "results_v2_full_zeroshot.csv", index=False)
# score(res_v2)


# Part 4. How did I get this running on the server? 

I created a separate project where I worked on this. It was as well done with uv. I downloaded the requisite package and ran the server through the command prompt. 

Because `/data/resource/` stores some data on platters rather than SSD, the cold start for vLLM was slow. It took almost 20 minutes to load into VRAM! So you will want to be careful of this if you run it yourself. 

**1. Check the GPUs and driver**
```bash
nvidia-smi
```

**2. Set environment variables** (add to `~/.bashrc`)
```bash
export HF_HUB_CACHE=/data/resource/huggingface/hub
export CUDA_DEVICE_ORDER=PCI_BUS_ID
```

**3. Create a separate environment for vLLM**
```bash
mkdir -p llm-judge/server && cd llm-judge/server
uv venv --python 3.12
```

**4. Install vLLM built for CUDA 12.9**
The driver on brains supports CUDA 12.3, and vLLM 0.33+ is CUDA 13 only, so we install the 0.32.0 `+cu129` wheel. List the available wheels:
```bash
curl -s "https://api.github.com/repos/vllm-project/vllm/releases?per_page=10" \
  | grep -o '"name": *"vllm-[^"]*+cu12[0-9][^"]*x86_64\.whl"'
```
Install it, using the exact filename from the list:
```bash
uv pip install "https://github.com/vllm-project/vllm/releases/download/v0.32.0/vllm-0.32.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" \
  --torch-backend=cu129
```

**5. Start the server inside tmux**
```bash
tmux new -s vllm
cd llm-judge/server
export VLLM_USE_FLASHINFER_SAMPLER=0
CUDA_VISIBLE_DEVICES=0 uv run vllm serve google/gemma-4-31B-it \
  --served-model-name gemma-4-31b \
  --host 127.0.0.1 --port 8000 \
  --max-model-len 8192 \
  --gpu-memory-utilization 0.92 \
  --reasoning-parser gemma4 2>&1 | tee gemma.log
```
Detach with **Ctrl+B, then D**. Reattach with `tmux attach -t vllm`. Loading takes several minutes; wait for "Application startup complete".

**6. Test from the terminal**
```bash
curl -s localhost:8000/v1/chat/completions -H "Content-Type: application/json" -d '{
  "model": "gemma-4-31b",
  "messages": [{"role": "user", "content": "Answer yes or no: is Paris in France?"}],
  "max_tokens": 5, "temperature": 0}'
```

**7. Connect from a notebook on brains**
```python
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")
r = client.chat.completions.create(
    model="gemma-4-31b",
    messages=[{"role": "user", "content": "Answer yes or no: is Paris in France?"}],
    max_tokens=5, temperature=0)
print(r.choices[0].message.content)
```

## Part 5. Your turn! 

**Part 5.1. Changing the prompt and seeing the difference.**

Notice that the prompt above is: "Does the following post from a film and TV question-and-answer site reveal a major plot point? Answer yes or no."

Try altering it to see if it makes a difference. Here are some options that I tried: 
- "Does the following post from a film and TV question-and-answer site reveal a major plot point aka a spoiler? Answer yes or no."
- "Consider the following post from a message board about film and television. Should the post be marked with a spoiler tag? Answer yes or no."
- "You are an expert film critic who has watched most every popular television and film show from recent memory. You are helping annotate a corpus of posts drawing upon your supreme cinematographic knowledge that would put Roger Ebert or Peter Travers to shame. Consider the following post from a message board about film and television. Should the post be marked with a spoiler tag? Answer yes or no.
  - The last one is an example of RCG (Role-Context-Guidance). This has been demonstrated in recent work to not be effective under almots all circumstances. It can even create noise. See for example: https://arxiv.org/abs/2603.18507 and https://arxiv.org/abs/2311.10054 despite the folk wisdom suggesting it works.

**Part 5.2. Changing the annotation structure**

Below you should be able to establish whether the LLM annotation can distinguish a question from an answer. Have a look at the code above for where we did the following: 
1. Ground truth label for question and answer
2. Check whether a sample is sufficiently balanced that one cannot simply detect this from word length.
3. Frankly this one should be super easy given that questions are likely to have '?', you might want to remove question marks or check for them the way that we checked for colons above. 